# 05 — Run the Assistant

[Notebook guide](README.md) · [Project overview](../README.md) · [English paper](../docs/en/paper.pdf)

Inspect retrieval and the recorded multi-turn question-answering examples.

| Item | Details |
| --- | --- |
| Inputs | Saved FAISS index and compatible merged model. |
| Outputs | Recorded retrieved passages and conversation outputs. |
| Requirements | Colab/Drive and GPU runtime with sufficient memory for the 8-bit model; trusted index metadata. |
| Configuration overview | `db_path` and `model_id`; retrieval k = 5; 8-bit model loading; max_new_tokens = 512; do_sample = False. History resets appear only at the starts of Sessions A and B. |

**Recorded experiment:** run code cells in their existing order. This presentation edit preserves executable code, parameters, prompts, data, and experimental outputs. Configuration is summarized here without moving assignments. Installation logs are omitted; execution counts belong to the original run. Serbian prompts, reference text, docstrings, and printed outputs remain verbatim.


## 1. Install dependencies and prepare the runtime

In [ ]:
!pip install -q langchain langchain-classic langchain-community langchain-huggingface \
    faiss-cpu transformers torch accelerate gradio sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
!pip install -q bitsandbytes

In [ ]:
import os
import torch
import warnings
import gradio as gr

from google.colab import drive

from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import PromptTemplate
import logging



/tmp/ipykernel_10583/2221723193.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
warnings.filterwarnings("ignore")

# Suppress bitsandbytes and autograd warnings
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)

# Suppress Hugging Face messages
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## 2. Set artifact paths and load retrieval

In [ ]:
db_path = "/content/drive/MyDrive/NLP_proyect/vector_database"
model_id = "/content/drive/MyDrive/NLP_proyect/merged_model"


In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2")
vector_db = FAISS.load_local(db_path, embedding_model, allow_dangerous_deserialization=True)
retriever = vector_db.as_retriever(search_kwargs={"k": 5})


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 3. Load the generator and decoding settings

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    low_cpu_mem_usage=True
)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'repetition_penalty', 'eos_token_id', 'temperature', 'no_repeat_ngram_size', 'do_sample', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.15,
    do_sample=False,
    return_full_text=False,
    repetition_penalty=1.05,
    no_repeat_ngram_size=4,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id
)
llm = HuggingFacePipeline(pipeline=pipe)


## 4. Assemble the prompt and conversation helpers

In [ ]:
contextualize_q_system_prompt = """Ukoliko postoji istorija razgovora i korisnik postavi pitanje koje se nadovezuje,
reformuliši to pitanje tako da bude samostalno i razumljivo bez istorije razgovora.
NEMOJ odgovarati na pitanje, samo ga reformuliši ako je potrebno, u suprotnom ga vrati identičnog."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_q_prompt)

system_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
1. Ti si stručni medicinski asistent i komuniciraš isključivo na srpskom jeziku.
2. Ako u kontekstu nema informacija potrebnih za odgovor, nemoj izmišljati nego jasno reci: "Nažalost, nemam te informacije u svojoj bazi podataka, najbolje se savetuj sa lekarom ili farmaceutom. Informacije sa interneta mogu biti lažne i štetne."
3. Piši svakodnevnim i lako shvatljivim jezikom.
4. Ukoliko je pitanje na latinici, odgovori na latinici. Ukoliko je pitanje na ćirilici, odgovori na ćirilici.
5. Na kraju svakog odgovora OBAVEZNO dodaj: "Pre upotrebe svakog medikamenta, obavezno se posavetuje sa lekarom ili farmaceutom. Dobijeni odgovor nije zamena za stručno mišljenje, već pomoćna alatka."

Pruženi medicinski kontekst:
{context}<|eot_id|><|start_header_id|>user<|end_header_id|>
{input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

qa_prompt = PromptTemplate(
    template=system_prompt,
    input_variables=["context", "input"]
)

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [ ]:
def rag_chat_funkcija(message, history):
    langchain_history = []
    for msg in history:
        role = msg.get("role")
        content = msg.get("content")
        if role == "user":
            langchain_history.append(HumanMessage(content=content))
        elif role == "assistant":
            langchain_history.append(AIMessage(content=content))

    rezultat = rag_chain.invoke({
        "input": message,
        "chat_history": langchain_history
    })
    return rezultat["answer"]


In [ ]:
import time

def pitaj(pitanje):
    t0 = time.time()
    rezultat = rag_chain.invoke({
        "input": pitanje,
        "chat_history": chat_history
    })
    trajanje = time.time() - t0
    odgovor = rezultat["answer"]
    chat_history.append(HumanMessage(content=pitanje))
    chat_history.append(AIMessage(content=odgovor))
    print(f"Q: {pitanje}\nA: {odgovor}\n[{trajanje:.1f}s]\n")
    return odgovor

## 5. Inspect retrieval before generation

In [ ]:
docs = retriever.invoke("Za šta se primenjuje lek Zoloft?")
for d in docs:
    print(d.metadata.get("izvor_fajl"), "\n", d.page_content, "\n---")

515-01-00535-22-001.pdf 
 Uzimanje leka Zoloft zajedno sa sledećim lekovima može prouzrokovati ozbiljna neželjena dejstva:
- Lekovi koji se nazivaju inhibitori monoaminooksidaze (MAOI), kao što je moklobemid (primenjuje se za lečenje depresije) i selegilin (primenjuje se za lečenje Parkinsonove bolesti), antibiotik linezolid i metilensko plavo (primenjuje se za terapiju povećane koncentracije methemoglobina u krvi). Ne primenjujte lek Zoloft zajedno sa ovim lekovima.  
- Lekovi koji se koriste za lečenje mentalnih poremećaja kao što je psihoza (pimozid). Ne primenjujte lek Zoloft zajedno sa pimozidom. 
---
515-01-00535-22-001.pdf 
 Šta sadrži lek Zoloft
Aktivna supstanca je sertralin.  
Jedna film tableta sadrži 50 mg ili 100 mg sertralina (u obliku sertralin-hidrohlorida).  
Pomoćne supstance: 
---
515-01-00535-22-001.pdf 
 1. Šta je lek Zoloft i čemu je namenjen
Lek Zoloft sadrži aktivnu supstancu sertralin. Sertralin spada u grupu lekova koji se nazivaju selektivni inhibitori preuzi

## 6. Session A: Dovicin in Latin script

**History reset:** the next cell starts a new conversation.

In [ ]:
chat_history = []


In [ ]:
pitaj("Kada ne smem da uzimam lek Dovicin?")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Kada ne smem da uzimam lek Dovicin?
A: 

Ne smete uzimat' lek Dovičin ukoliko:

- ste alergičan (preosvetljiv) na dokičiklin ili nekli drugi leik iz grupe tetraciklina, ili na bili koju od pomoći supstancija ovog leika (navedena u odelju 6);
- ste trudne ili planirate roditeljstvo;
- dojite;
- imate obolenje praćenim otežnim gutanjem.

Pre upotrebu svakog leka obavezno saovetujte se sa lekarim ili farmaceutom.
[26.6s]



"\n\nNe smete uzimat' lek Dovičin ukoliko:\n\n- ste alergičan (preosvetljiv) na dokičiklin ili nekli drugi leik iz grupe tetraciklina, ili na bili koju od pomoći supstancija ovog leika (navedena u odelju 6);\n- ste trudne ili planirate roditeljstvo;\n- dojite;\n- imate obolenje praćenim otežnim gutanjem.\n\nPre upotrebu svakog leka obavezno saovetujte se sa lekarim ili farmaceutom."

In [ ]:
pitaj("A šta ako uzimam i drugi lek pored Dovicina? Šta da radim?")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: A šta ako uzimam i drugi lek pored Dovicina? Šta da radim?
A: 

Ako uzimate i drugi lijek pored Dovicina, važno je da obavijestite svoga liječnika ili farmaceutu o tome. Lekovi mogu uticati jedni na druge, što može uticati na njihovo dejstvo ili povećavati rizik od nepoželjnih reakcija.

U slučaju Dovicina postoje neke interakcija sa drugim lijekovima koje treba uzeti u obzir:

* Antacidi (leki za smanjivanje kiseonika u želucu): Neka antacidi mogu smanjivati apsorpciju Dovicina. Preporučuje se da se uzimati antacidi na različito vreme od Dovicina.
* Karbamazepina, fenitona i barbituratima (leki koji se korisne za lečenja epilepsije): Dovacin može povišati nivo karbamazepam, fenitonske i barbiturate u krvi. Vaš liječnik može morati da poveća doze ovih lijekova ili da promijeni režim terapije.
* Varfarin i kumarin (leki protiv krvarenja): Dovcin može poništiti dejstvo varfarnina i kumarina. Vaš lijekar može mora da poveži frekvenciju testiranja krvnih testova za krvarenje.
* Oralni k

'\n\nAko uzimate i drugi lijek pored Dovicina, važno je da obavijestite svoga liječnika ili farmaceutu o tome. Lekovi mogu uticati jedni na druge, što može uticati na njihovo dejstvo ili povećavati rizik od nepoželjnih reakcija.\n\nU slučaju Dovicina postoje neke interakcija sa drugim lijekovima koje treba uzeti u obzir:\n\n* Antacidi (leki za smanjivanje kiseonika u želucu): Neka antacidi mogu smanjivati apsorpciju Dovicina. Preporučuje se da se uzimati antacidi na različito vreme od Dovicina.\n* Karbamazepina, fenitona i barbituratima (leki koji se korisne za lečenja epilepsije): Dovacin može povišati nivo karbamazepam, fenitonske i barbiturate u krvi. Vaš liječnik može morati da poveća doze ovih lijekova ili da promijeni režim terapije.\n* Varfarin i kumarin (leki protiv krvarenja): Dovcin može poništiti dejstvo varfarnina i kumarina. Vaš lijekar može mora da poveži frekvenciju testiranja krvnih testova za krvarenje.\n* Oralni kontracepcijski tableti: Dovcin smanjuju efikasnoću oral

### Session A continued: Dovicin in Cyrillic

The original notebook does not reset history here. Earlier turns remain available.

In [ ]:
pitaj("Када не смем да узимам лек Dovicin?")

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Када не смем да узимам лек Dovicin?
A: 

Nemojte uzimati Dovicin ako ste alergijski osjetljivi na doksikiclin ili neke druge lekove iz grupe tetraciklina, ili ako ste alerhgijski osetljivi na bilo kakvu od pomoći ovog lekove.
[23.0s]



'\n\nNemojte uzimati Dovicin ako ste alergijski osjetljivi na doksikiclin ili neke druge lekove iz grupe tetraciklina, ili ako ste alerhgijski osetljivi na bilo kakvu od pomoći ovog lekove.'

In [ ]:
pitaj("Зашто си ми генерисао одговор на латиници?")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Зашто си ми генерисао одговор на латиници?
A: 
Извините, ако сам генерисирао одговор у латиницама. Као АИ, могу да генеришем текст на било ком језику, укључувајући и српски. Међутим, када сам примио питање на српском, могао сам да генеришам одговор на срспком. Једна могућност је што сам грешка генерисаног одговора била због ограничења мојих алгоритама или података. Ако желите, могу да покушам да поново генеришем одговор на ваше питаѪ на србском.
[57.5s]



'\nИзвините, ако сам генерисирао одговор у латиницама. Као АИ, могу да генеришем текст на било ком језику, укључувајући и српски. Међутим, када сам примио питање на српском, могао сам да генеришам одговор на срспком. Једна могућност је што сам грешка генерисаног одговора била због ограничења мојих алгоритама или података. Ако желите, могу да покушам да поново генеришем одговор на ваше питаѪ на србском.'

In [ ]:
pitaj("Али већ си генерисао претходни одговор на српском. Генериши одговор на ћирилици.")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Али већ си генерисао претходни договор на српском. Генериши одговор на ћирилици.
A: 

Могућа незелена дејстава
Као и сви леци, оваи лек може дати нежелена дељства, и ако се не мора да се појаве код свијех пацијенатех који га узимајух.  
Узимање овог лека може изазвати алергиичну реакцију, посебно код особа које су алергичне на лекове из групе АЦЕ инхибитора. Ове реакцијех могу укључивати оток, осип, оток грла, једа, уста, дисаћа, осип по кожи, свраб, крвоточење из носа, осећаће да је дисаёе тешко, осецање да ће се удати, осеќања да једе тешки, осецяње слабости, осеције да људи не могу да разумеју шта говорите, осеоције да се осећате да ћете западнути, осеочању да сањате, осеосећаље да сајете, осеоскопењу, осеокање у глави, осеокопеће, осеопећађе, осеоцежењ, осеожаљењ.
[112.3s]



'\n\nМогућа незелена дејстава\nКао и сви леци, оваи лек може дати нежелена дељства, и ако се не мора да се појаве код свијех пацијенатех који га узимајух.  \nУзимање овог лека може изазвати алергиичну реакцију, посебно код особа које су алергичне на лекове из групе АЦЕ инхибитора. Ове реакцијех могу укључивати оток, осип, оток грла, једа, уста, дисаћа, осип по кожи, свраб, крвоточење из носа, осећаће да је дисаёе тешко, осецање да ће се удати, осеќања да једе тешки, осецяње слабости, осеције да људи не могу да разумеју шта говорите, осеоције да се осећате да ћете западнути, осеочању да сањате, осеосећаље да сајете, осеоскопењу, осеокање у глави, осеокопеће, осеопећађе, осеоцежењ, осеожаљењ.'

### Session A continued: Nimulid

This block continues the same history; it is not an isolated test.

In [ ]:
pitaj("Која је уобичајена доза за лек Nimulid?")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Која је уобичајена доза за лек Nimulid?
A: 
Уобићајна доза лека Nimulid је једна таблета (100 мг) два пута дневно, након оброка.
[102.1s]



'\nУобићајна доза лека Nimulid је једна таблета (100 мг) два пута дневно, након оброка.'

In [ ]:
pitaj("Мислим да сам случајно прекорачио дозу. Узео сам по једну таблету више сваки пут. Шта ми се може десити?")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Мислим да сам случајно прекорачио дозу. Узео сам по једну таблету више сваки пут. Шта ми се може десити?
A: 
Ако сте узели више од прописане дозе, одмах разговарати са својим лекаром или одмах идите до најближег здравственог центра.
[105.3s]



'\nАко сте узели више од прописане дозе, одмах разговарати са својим лекаром или одмах идите до најближег здравственог центра.'

In [ ]:
pitaj("Такође сам конзумирао алкохол. Да ли ми се нешто може десити?")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Такође сам конзумирао алкохол. Да ли ми се нешто може десити?
A: 
Ако стручној помоћи не можете да добијете, обавести свог лекара о свим лековима које узимате, укључући алкохолне пиће.
[113.2s]



'\nАко стручној помоћи не можете да добијете, обавести свог лекара о свим лековима које узимате, укључући алкохолне пиће.'

## 7. Session B: Pressing

**History reset:** the next cell starts a new conversation.

In [ ]:
chat_history = []


In [ ]:
pitaj("Da li deca smeju da konzumiraju Pressing?")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Da li deca smeju da konzumiraju Pressing?
A: 

Nažalosti, nemam informaciju o dozama i upotrebi leka Pressing kod dece. Najbolje je da se savetujete sa lekarjem ili farmaceutima pre upotrebe bilo kog leka, posebno ako je pacijenat deca.
[11.5s]



'\n\nNažalosti, nemam informaciju o dozama i upotrebi leka Pressing kod dece. Najbolje je da se savetujete sa lekarjem ili farmaceutima pre upotrebe bilo kog leka, posebno ako je pacijenat deca.'

In [ ]:
pitaj("Da li odrasle osobe mogu da konzumiraju Pressing?")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Da li odrasle osobe mogu da konzumiraju Pressing?
A: 
Da bi se odredio da li odraslom osobi dozvoljava da uzima lek Crestora, potrebno je da se razmotri njena medicinska istorija i da se proceni da li ima bilo kakva protivindikacija za uzimanje ovog lekova.
[30.2s]



'\nDa bi se odredio da li odraslom osobi dozvoljava da uzima lek Crestora, potrebno je da se razmotri njena medicinska istorija i da se proceni da li ima bilo kakva protivindikacija za uzimanje ovog lekova.'

In [ ]:
pitaj("Ali ja te ne pitam za lek Crestora, već za lek Pressing?")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Ali ja te ne pitam za lek Crestora, već za lek Pressing?
A: 

Žao mi je, ali nisam imao informacije o leku Pressing. Da li možete da mi dajete više detalja o leku, kao što je njegova namena, upotreba i neželena dejstva?
[39.3s]



'\n\nŽao mi je, ali nisam imao informacije o leku Pressing. Da li možete da mi dajete više detalja o leku, kao što je njegova namena, upotreba i neželena dejstva?'

### Session B continued: more specific wording

History is retained while the wording of the question changes.

In [ ]:
pitaj("Da li deca smeju da konzumiraju Pressing sirup?")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Da li deca smeju da konzumiraju Pressing sirup?
A: 

Nažalosti, nemam informacije o dozama i dozvoljenim uzimanju leka Pressin sirup za decu. Pre upotrebe bilo kog leka, obavezni ste da se posavete sa lekarem ili farmaceutičkim tehničarom.
[10.9s]



'\n\nNažalosti, nemam informacije o dozama i dozvoljenim uzimanju leka Pressin sirup za decu. Pre upotrebe bilo kog leka, obavezni ste da se posavete sa lekarem ili farmaceutičkim tehničarom.'

In [ ]:
pitaj("Šta je to Pressing sirup?")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Šta je to Pressing sirup?
A: 

Pressing sirup je lek koji sadrži loratadin, koji pripada grupi antihistamina (blokatora H1 receptara). Koristi se za ublaženje simptama povezanim sa alergijom, kao što su kihanje, curenja iz nosa i svrab.
[40.9s]



'\n\nPressing sirup je lek koji sadrži loratadin, koji pripada grupi antihistamina (blokatora H1 receptara). Koristi se za ublaženje simptama povezanim sa alergijom, kao što su kihanje, curenja iz nosa i svrab.'

In [ ]:
pitaj("Da li deca smeju da konzumiraju Pressing sirup?")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Da li deca smeju da konzumiraju Pressing sirup?
A: 

Ne, deca mlađa od 2 године не сме да узимају лек Pressing sirуп.
[34.3s]



'\n\nNe, deca mlađa od 2 године не сме да узимају лек Pressing sirуп.'

In [ ]:
pitaj("A deca starija od 2 godine?")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: A deca starija od 2 godine?
A: 

Deca i adolescenati stariji od 2 i mlađeg od 18. godine: Lek Crestor u dozama od 40mg ne sme se primenjvati.
[94.6s]



'\n\nDeca i adolescenati stariji od 2 i mlađeg od 18. godine: Lek Crestor u dozama od 40mg ne sme se primenjvati.'

### Session B continued: repeated standalone wording

Although the question is self-contained, the original code does not reset conversation history.

In [ ]:
pitaj("Šta je to Pressing sirup?")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Šta je to Pressing sirup?
A: 

Pressing sirup je lek koji sadrži loratadin, koji pripada grupi antihistamina. Koristi se za ublaženje simptome alergiјskih reakcija, kao što se javljaju kod alergije na polen, i za lečenju hroničnog idiopatskog urtikariјa.
[43.8s]



'\n\nPressing sirup je lek koji sadrži loratadin, koji pripada grupi antihistamina. Koristi se za ublaženje simptome alergiјskih reakcija, kao što se javljaju kod alergije na polen, i za lečenju hroničnog idiopatskog urtikariјa.'

### Session B continued: Zoloft

The existing history remains in use.

In [ ]:
pitaj("Za šta se primenjuje lek Zoloft")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Za šta se primenjuje lek Zoloft
A: 

Lek Zolof se može koristiti za lečenju sledećih stanja:

- Depresija i prevencija ponovnoga javljanje depresiji kod odrasle populacije.
- Socijalni poremečaj kod odrasnih osoba.
- Posttraumatiski stresni sindrom kod odraslaca.
- Paničan sindrom kod zdravih odraslog stanovništva.
- Opsesivo-kompulzijski sindrom kod pacijenata od 6 godina starosti i više.
[31.6s]



'\n\nLek Zolof se može koristiti za lečenju sledećih stanja:\n\n- Depresija i prevencija ponovnoga javljanje depresiji kod odrasle populacije.\n- Socijalni poremečaj kod odrasnih osoba.\n- Posttraumatiski stresni sindrom kod odraslaca.\n- Paničan sindrom kod zdravih odraslog stanovništva.\n- Opsesivo-kompulzijski sindrom kod pacijenata od 6 godina starosti i više.'

In [ ]:
pitaj("Započeo sam terapiju ovim lekom, da li je normalno što ponekad pomislim o samopovređivanju?")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Započeo sam terapiju ovim lekom, da li je normalno što ponekad pomislim o samopovređivanju?
A: 

Nažalosti, nemam informacije o vašoj ličnoj situaciji, ali mogu da vas uputim na generalne informacije. Ponekad, kada počnete da uzimate antidepresive, mogu se javiti misle o samopovećanju ili samopoveđivanju zbog toka leka. Ovo je normalno, ali je važno da se obratim Vašem lekaru ili u bolnicom ako se javljaju uznemirene misli ili da se Vaši simptomi pogoršaju.
[115.9s]



'\n\nNažalosti, nemam informacije o vašoj ličnoj situaciji, ali mogu da vas uputim na generalne informacije. Ponekad, kada počnete da uzimate antidepresive, mogu se javiti misle o samopovećanju ili samopoveđivanju zbog toka leka. Ovo je normalno, ali je važno da se obratim Vašem lekaru ili u bolnicom ako se javljaju uznemirene misli ili da se Vaši simptomi pogoršaju.'

In [ ]:
pitaj("Dobro. Hvala ti. A koliko je antideprisivima potrebno da deluju?")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Dobro. Hvala ti. A koliko je antideprisivima potrebno da deluju?
A: 

Antidepresivi su lekovi koji se koriste za lečenje depresije i drugih mentalnih zdravljnih stanja. Vreme koje je potrebno za njihovo delovanje varira u zavisnosti od vrste leka i pojedinačnih faktora, kao što su doza, starost i zdravsto stanje pacijenta.

Neki antidepresivi počinju da deluju u roku od nekoliko sedmica, dok drugi mogu potrebno vreme da počnu da pokazuju svoje efekte. Generalno, antidepresive se mogu deliti na tri grupe:

1. Triciklični antidepresanti: Ovi lekovi počinju generalno da delujeju u roku od 2-4 nedelje, ali može potrebno nekoliko meseci da se postigne maksimalna efikasnost.
2. Sekundarni amfetaminski antidepresani: Ovi počinju u proseku da deluju za 1-2 nedelje i mogu dostići svoju maksimalnu efikasnoću u roku od mesec dana.
3. Selectivni inhibitori ponovnog ugradnja serotonina (SSRI): Ovi počnu da deluju generalno u roku od nedelju do dva meseca i mogu potencijalno da dostignu svoju mak

'\n\nAntidepresivi su lekovi koji se koriste za lečenje depresije i drugih mentalnih zdravljnih stanja. Vreme koje je potrebno za njihovo delovanje varira u zavisnosti od vrste leka i pojedinačnih faktora, kao što su doza, starost i zdravsto stanje pacijenta.\n\nNeki antidepresivi počinju da deluju u roku od nekoliko sedmica, dok drugi mogu potrebno vreme da počnu da pokazuju svoje efekte. Generalno, antidepresive se mogu deliti na tri grupe:\n\n1. Triciklični antidepresanti: Ovi lekovi počinju generalno da delujeju u roku od 2-4 nedelje, ali može potrebno nekoliko meseci da se postigne maksimalna efikasnost.\n2. Sekundarni amfetaminski antidepresani: Ovi počinju u proseku da deluju za 1-2 nedelje i mogu dostići svoju maksimalnu efikasnoću u roku od mesec dana.\n3. Selectivni inhibitori ponovnog ugradnja serotonina (SSRI): Ovi počnu da deluju generalno u roku od nedelju do dva meseca i mogu potencijalno da dostignu svoju maksimulnu efikasnost u roku od tri do šest meseci.\n\nSve u svem

In [ ]:
pitaj("Dobro. Hvala ti na pomoći")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Dobro. Hvala ti na pomoći
A: 
[39.6s]



''

### Session B continued: out-of-domain and English questions

The original inputs and generated responses are reproduced unchanged.

In [ ]:
pitaj("Kaži mi šta je to okean?")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Kaži mi šta je to okean?
A: 

Okean je veliki ocean koji pokriva oko 71% površine Zemlje.
[11.1s]



'\n\nOkean je veliki ocean koji pokriva oko 71% površine Zemlje.'

In [ ]:
pitaj("Do you have an ability to talk in English?")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Do you have an ability to talk in English?
A: 

I can understand and respond in English, but I'm a Serbian language model, so my primary language is Serbian. If you'd like, we can communicate in English, and I'll do my best to provide accurate and helpful responses.
[15.7s]



"\n\nI can understand and respond in English, but I'm a Serbian language model, so my primary language is Serbian. If you'd like, we can communicate in English, and I'll do my best to provide accurate and helpful responses."

In [ ]:
pitaj("Can I drive a car if I take the medicin  e called Paracetamol?")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Can I drive a car if I take the medicine called Paracetamol?
A: 

Da li možete vožditi auto dok uzimate lek Parasetamol?

Nije utvrdjeno da lekovi Paracetamola utiču na vašu sposobnost da upravljujete vozilom. Međutim, lek Paracetalom može izdati vrtoglas i pospanos, što može utiči na vaša sposobnost vožnje. Ako se osećate vrtoglava ili pospanog dok uzimate lek Parcetamol, ne voždite vozilo.
[35.6s]



'\n\nDa li možete vožditi auto dok uzimate lek Parasetamol?\n\nNije utvrdjeno da lekovi Paracetamola utiču na vašu sposobnost da upravljujete vozilom. Međutim, lek Paracetalom može izdati vrtoglas i pospanos, što može utiči na vaša sposobnost vožnje. Ako se osećate vrtoglava ili pospanog dok uzimate lek Parcetamol, ne voždite vozilo.'

## Reading the recorded results

Read outputs as experimental observations, including errors and failed instruction following. The session labels describe the existing reset behavior; no reset or test call was added. Recorded answers are not medical guidance.